# 同季度EPS拆股与比较值核验
只采用相应报告中已有证据的股票拆分，不用价格分红复权。遍历所有可配对年度同季度；区间仅相切与不相交均列为待审，不覆盖原报值。

In [1]:
from pathlib import Path
import os,json,hashlib,re
import pandas as pd,qlib
from qlib.workflow import R
from bs4 import BeautifulSoup
from src.workflow.minute_action_value import _source
from src.data.nvda_seasonal_eps import split_bridge,split_ratio_between_reports
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/nvda_eps_split_ledger_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/nvda_eps_corpus_20261015_v1';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
source,audit=_source(root,p['source'],['analysis/events','inputs/raw_html']);events={e['fiscal_quarter']:e for e in source['analysis/events']};ledger=[dict(e) for e in p['split_events']]
for event in ledger:
 record=events[event['known_in_report']];payload=source['inputs/raw_html'][event['payload']];assert hashlib.sha256(payload).hexdigest()==record['source_sha256'];assert record['release_date']==event['known_by_date'];text=BeautifulSoup(payload,'html.parser').get_text(' ',strip=True)
 for term in event['evidence_terms']:assert term in text
 event['source_url']=record['source_url'];event['source_sha256']=record['source_sha256']
review=json.loads((card/'restatement_review.json').read_text());existing=json.loads((card/'summary.json').read_text())
from src.data.nas_market_data import materialize_nas_evidence
if not Path(existing['recorder_local_dir']).exists():materialize_nas_evidence(card/'nas_binding.json',card=card)
with R.uri_context(str(card/'qlib_recorder')):
 prior=R.get_recorder(recorder_id=existing['recorder_id'],experiment_id=existing['experiment_id']);pdf=prior.load_object('inputs/restatement_pdf');stored_review=prior.load_object('inputs/restatement_review')
assert hashlib.sha256(pdf).hexdigest()==review['source']['sha256'] and review==stored_review
for adjustment in review['adjustments']:
 assert events[adjustment['quarter']]['current_eps']==adjustment['as_reported']
 assert events[adjustment['next_year_report']]['prior_year_comparative_eps']==adjustment['as_adjusted']
rows=[]
for quarter,current in sorted(events.items()):
 match=re.fullmatch(r'FY(\d{4})Q([1-4])',quarter);previous_key=f'FY{int(match.group(1))-1}Q{match.group(2)}'
 if previous_key not in events:rows.append({'current_quarter':quarter,'previous_quarter':previous_key,'status':'needs_prior_original_report'});continue
 previous=events[previous_key];ratio=split_ratio_between_reports(previous,current,ledger);result=split_bridge(previous,current,split_ratio=ratio);rows.append({**result,'status':'rounding_compatible' if result['rounding_relation']=='overlap' else 'needs_restatement_review','current_release_date':current['release_date'],'previous_release_date':previous['release_date'],'previous_source_sha256':previous['source_sha256'],'current_source_sha256':current['source_sha256']})
table=pd.DataFrame(rows)
for adjustment in review['adjustments']:
 mask=table.current_quarter.eq(adjustment['next_year_report']);assert int(mask.sum())==1 and table.loc[mask,'status'].iloc[0]=='needs_restatement_review';table.loc[mask,'status']='explained_accounting_adjustment'
valid=table[table.status.ne('needs_prior_original_report')];counts=table.status.value_counts().to_dict();policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/bridges']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder'),recorder_id=existing['recorder_id'],resume=True):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_audit=audit,split_ledger=ledger,code=(root/'src/data/nvda_seasonal_eps.py').read_text());rec.save_objects(artifact_path='analysis',bridges=table);rec.log_metrics(available_bridges=len(valid),unexplained_bridges=int(table.status.eq('needs_restatement_review').sum()),registered_artifact_count=sum(f.is_file() for f in (card/'qlib_recorder'/str(rec.experiment_id)/rec.id/'artifacts').rglob('*')))
summary={'experiment':card.name,'decision':'needs_vintage_and_clock_qualification','new_fits':0,'new_backtests':0,'source_reports':len(events),'available_year_over_year_bridges':len(valid),'status_counts':counts,'training_input_qualified':False,'global_split_ledger_completeness_certified':False,'provider_uri':str(provider),'region':'us','as_of':'2026-10-14','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False}
rec.save_objects(artifact_path='analysis',summary=summary)
rec.log_metrics(registered_artifact_count=sum(f.is_file() for f in (card/'qlib_recorder'/str(rec.experiment_id)/rec.id/'artifacts').rglob('*')))
table.to_csv(card/'quarter_bridges.csv',index=False);pd.DataFrame(ledger).drop(columns=['evidence_terms']).to_csv(card/'split_ledger.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));binding=json.loads((root/p['source']['card']/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2));a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(counts);print(table[table.status.eq('needs_restatement_review')].to_string(index=False))


# 季节性EPS拆股及会计调整：数值差异已解释，训练资格仍未通过

对49份已解析报告建立45组相邻财年同季度桥接，43组按拆股和显示精度相容，2组已由官方2017年报的会计调整表解释；另4份缺上一年原始报告，未补造。0拟合、0回测，没有给既有策略增加收益或替换生产模型。

两项拆股账本仅用于EPS股份口径：2021年4拆1、2024年10拆1。账本保留生效日期、当前报告可确认的日期、原报告哈希；只应用两份报告之间且截至当前报告已有证据的拆股，拒绝未来证据、重复事件、同日时序未明和现金分红调整。2024年6月7日拆股完成与6月10日拆后交易不是同一概念；本账本处理报告每股口径，不是行情复权。

|历史季度|原报告GAAP diluted EPS|后来调整值|下一年比较栏|
|---|---:|---:|---:|
|FY2017Q1|0.33|0.35|0.35|
|FY2017Q2|0.40|0.41|0.41|

NVDA 2017年报Note18（PDF第175页、印刷页93）明确列出as reported/as adjusted：FY2017Q3采用股票薪酬会计准则，并调整该财年前两季。原Q1差异不是拆股或舍入；Q2显示舍入区间仅边界相切，也不能当作相容通过。PDF表格已视觉核对，季度与半年列分开；SEC同一10-K文本交叉一致。原始报告值0.33/0.40保留，绝不将后来的0.35/0.41回填早期样本。

来源：发行人CDN年报 https://s201.q4cdn.com/141608511/files/doc_financials/annual/2017/NVIDIA_2017AnnualReview_Proxy_10-K.pdf ，及SEC https://www.sec.gov/Archives/edgar/data/1045810/000104581017000027/nvda-2017x10k.htm 。本地直接SEC下载403；实际保留字节来自发行人CDN的HTTP200 PDF，不把网页工具读到的SEC文本冒充已下载SEC原件。PDF、转录和哈希在Recorder/NAS，来源日期与当前下载时间分开。

这是会计差异的解释，不是完整历史版本认证。报告当前副本不等于当时首次发布字节，账本不是完整全部拆股事件证明；会计调整的首次公开时间仍需确认，精确发布时钟及日内可交易时点也未认证。即便采用保守延迟，也不能自动修复历史值被后期改写的风险。季节性盈利变化不是分析师一致预期surprise。

决定needs_vintage_and_clock_qualification，training_input_qualified=false，不计算SUE、不按未来收益挑季度、不训练未认证特征。下一具体缺项是季度原始版本及调整首次可用时点；必须明确原报告系列与当期可比系列的用途，不能混合后拟合。来源问题未解决前不重复已有财报预期失败路线。

4项既有及本轮扩展合同测试通过。修复了notebook派生拆股元数据对原plan对象的别名修改，Recorder inputs/plan与根目录冻结plan逐值一致；45桥接原始数值不变。复用同一Recorder身份，NAS冷恢复逐对象哈希通过。
